In [1]:
import torch
import cv2
import numpy as np
from collections import deque

In [2]:
# import warnings
# warnings.filterwarnings("ignore", category=FutureWarning)

In [9]:
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# Initialize video capture for the video file
vs = cv2.VideoCapture('shooting.mp4')  # Change 'shooting2.mp4' to your video file

# Deque to store tracked points
pts = deque(maxlen=64)

while True:
    # Read the next frame from the video
    ret, frame = vs.read()

    # Break the loop if there are no frames left
    if not ret:
        break

    # Resize frame for consistent processing
    frame = cv2.resize(frame, (854, 480))

    # Perform YOLO inference
    results = model(frame)

    # Parse results
    for det in results.xyxy[0]:
        x1, y1, x2, y2, conf, cls = det
        label = results.names[int(cls)]

        # Filter results for basketball only (e.g., "sports ball" class)
        if label == 'sports ball':  # Adjust based on the YOLOv5 model class
            # Draw bounding box
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)

            # Compute the center of the bounding box
            center = (int((x1 + x2) / 2), int((y1 + y2) / 2))
            cv2.circle(frame, center, 5, (0, 0, 255), -1)

            # Update the points queue
            pts.appendleft(center)

    # Draw lines between tracked points
    for i in range(1, len(pts)):
        if pts[i - 1] is None or pts[i] is None:
            continue
        thickness = int(np.sqrt(64 / float(i + 1)) * 2.5)
        cv2.line(frame, pts[i - 1], pts[i], (0, 255, 255), thickness)

    # Display the frame
    cv2.imshow('Frame', frame)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

# Cleanup resources
vs.release()
cv2.destroyAllWindows()

Using cache found in /Users/me/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-12-3 Python-3.12.0 torch-2.5.1 CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
/Users/me/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:892: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/me/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:892: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/me/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:892: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/Users/me/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:892: F